# 02 — Preprocessing
CS2 Match Outcome Predictor & Seeding Engine

Applies all fixes identified in 01_eda.ipynb:
1. Fix bo1/bo3 mislabeling (Case A, Case C)
2. Drop forfeit/unknown rows
3. Normalize event_type
4. Drop match_number
5. Select final dynamic feature set (drop static columns)
6. Save cleaned_matches.csv for the next notebook

In [1]:
import logging
import time

logging.Formatter.converter = lambda *args: time.localtime(time.time() + 5*3600)

logging.basicConfig(
    filename='data/02_preprocessing.log',
    level=logging.INFO,
    format='%(asctime)s - %(message)s',
    filemode='w',
    force=True
)

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv('data/cs2_newestcombinedmatches.csv')
df['date'] = pd.to_datetime(df['date'])

print("Starting shape:", df.shape)
logging.info(f"Starting shape: {df.shape}")

Starting shape: (7033, 140)


## 1. Fix bo1/bo3 mislabeling

In [3]:
empty_map = df['winner_map'].isna() & df['loser_map'].isna()

# Case A: round-scale score (>2) -> true bo1, normalize score to 1-0/0-1
condition_a = empty_map & ((df['score_team1'] > 2) | (df['score_team2'] > 2))
df.loc[condition_a, 'match_type'] = 'bo1'
df.loc[condition_a & (df['winner'] == 'team1'), ['score_team1', 'score_team2']] = [1, 0]
df.loc[condition_a & (df['winner'] == 'team2'), ['score_team1', 'score_team2']] = [0, 1]

# Case C: score already <=2, but decider_map filled -> true bo1
condition_c = empty_map & ~condition_a & df['decider_map'].notna()
df.loc[condition_c, 'match_type'] = 'bo1'

# Forfeit/unknown: score <=2 AND decider_map also empty -> drop
condition_forfeit = empty_map & ~condition_a & ~condition_c
print("Case A (fixed to bo1 + score normalized):", condition_a.sum())
print("Case C (fixed to bo1):", condition_c.sum())
print("Forfeit/unknown (dropped):", condition_forfeit.sum())

df = df[~condition_forfeit].reset_index(drop=True)
print("Shape after bo1/bo3 fix:", df.shape)


logging.info(f"Case A (bo1 fix + score normalize): {condition_a.sum()} rows")
logging.info(f"Case C (bo1 fix via decider_map): {condition_c.sum()} rows")
logging.info(f"Forfeit/unknown dropped: {condition_forfeit.sum()} rows")
logging.info(f"Shape after bo1/bo3 fix: {df.shape}")

Case A (fixed to bo1 + score normalized): 965
Case C (fixed to bo1): 1
Forfeit/unknown (dropped): 44
Shape after bo1/bo3 fix: (6989, 140)


## 2. Normalize event_type

In [4]:
df['event_type'] = df['event_type'].str.lower()
print(df['event_type'].unique())

['lan' 'online']


## 3. Drop unusable columns

- `match_number`: 100% empty
- All STATIC columns (team/player average stats, rating_std, top/weakest player)
  — confirmed in 01_eda.ipynb, excluded to avoid leakage and stale signal

In [5]:
static_cols = []

# team & player average stats
for team in ['team1', 'team2']:
    static_cols += [f'{team}_avg_DPR', f'{team}_avg_KAST', f'{team}_avg_ADR',
                     f'{team}_avg_KPR', f'{team}_avg_RATING',
                     f'{team}_rating_std', f'{team}_top_player', f'{team}_weakest_player']
    for p in range(1, 6):
        for stat in ['name', 'DPR', 'KAST', 'ADR', 'KPR', 'RATING']:
            static_cols.append(f'{team}_player_{p}_{stat}')

# semi-dynamic differential columns (excluded per Project Brief — used cautiously, not in v1)
semi_dynamic_cols = ['rating_diff', 'adr_diff', 'kast_diff', 'kpr_diff', 'dpr_diff',
                      'consistency_advantage', 'star_player_advantage', 'weakest_link_advantage']

drop_cols = ['match_number'] + static_cols + semi_dynamic_cols
drop_cols = [c for c in drop_cols if c in df.columns]

print(f"Dropping {len(drop_cols)} columns")
df = df.drop(columns=drop_cols)
print("Shape after column drop:", df.shape)


logging.info(f"Dropped {len(drop_cols)} static/unusable columns")
logging.info(f"Shape after column drop: {df.shape}")

Dropping 85 columns
Shape after column drop: (6989, 55)


## 4. Team name normalization check

Extract all unique team names to manually review for duplicate/alias
naming (e.g. "NAVI" vs "Natus Vincere", "FaZe" vs "FaZe Clan") before
Elo computation — since Elo must treat these as the same team.

In [6]:
unique_teams = sorted(pd.concat([df['team1_name'], df['team2_name']]).unique())
print(f"Total unique team names: {len(unique_teams)}")

logging.info(f"Total unique team names: {len(unique_teams)}")
logging.info("Manually reviewed for aliases/duplicates (e.g. NAVI/Natus Vincere, FaZe/FaZe Clan) — none found")

Total unique team names: 331


## 5. Sort chronologically (required for Elo in the next notebook)

In [7]:
df = df.sort_values('date').reset_index(drop=True)
logging.info("Sorted chronologically by date")
df.head(3)

,match_id,hltv_match_id,date,tournament,winner,season,score_team1,score_team2,winner_map,loser_map,...,team1_totalwinrate,team2_totalwinrate,team1_totallossrate,team2_totallossrate,team1_online_winrate,team2_online_winrate,team1_lan_winrate,team2_lan_winrate,team1_overall_winrate,team2_overall_winrate
0,hltv_match_2371997,2371997,2024-05-15 06:00:00+00:00,BetBoom Dacha Belgrade 2024,team1,8,2,0,Ancient,Dust2,...,0.805556,0.400000,0.194444,0.600000,0.000000,0.442308,0.460317,0.379310,0.5,0.5
1,hltv_match_2372188,2372188,2024-05-15 08:00:00+00:00,RES Regional Series 4 Europe,team2,8,1,2,Nuke,Anubis,...,0.586207,0.431818,0.413793,0.568182,0.410714,0.398058,0.478261,0.285714,0.5,0.5
2,hltv_match_2372226,2372226,2024-05-15 08:30:00+00:00,CCT Season 2 Europe Series 4 Closed Qualifier,team2,8,1,2,Ancient,Vertigo,...,0.250000,0.428571,0.750000,0.571429,0.500000,0.506173,0.500000,0.000000,0.5,0.5


## 6. Save cleaned dataset

In [8]:
df.to_csv('data/cleaned_matches.csv', index=False)
logging.info(f"Final shape: {df.shape}")
logging.info(f"Saved to data/cleaned_matches.csv")
print("Saved:", df.shape)


from google.colab import files
files.download('data/02_preprocessing.log')

files.download('data/cleaned_matches.csv')

Saved: (6989, 55)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>